# CO 370 — LP Relaxation & Shadow Prices

1. Solve the MIP with relaxed bounds
2. Relax all integer/binary variables to continuous (LP relaxation)
3. Compute and interpret shadow prices on nutritional and structural constraints
4. Report integrality gap

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

from model import load_data, build_params, build_model, NUT_BOUNDS_RELAXED, NUTRIENTS, DAYS

os.makedirs("results", exist_ok=True)

In [ ]:
prices, meals, meal_ings, meal_nuts = load_data()
ingredients, meal_ids, B, L, D, p, s, r, a = build_params(
    prices, meals, meal_ings, meal_nuts
)
lam = 0.5
print(f"{len(meal_ids)} meals | {len(ingredients)} ingredients | lam={lam}")

## Solve MIP

In [ ]:
m_mip, x, y, w = build_model(
    ingredients, meal_ids, B, L, D, p, s, r, a,
    lam=lam, nut_bounds=NUT_BOUNDS_RELAXED
)
m_mip.setParam("OutputFlag", 0)
m_mip.optimize()
mip_obj = m_mip.ObjVal
print(f"MIP objective: ${mip_obj:.4f}")

## LP Relaxation

Use `Model.relax()` to drop all integrality constraints. Shadow prices (duals) are only defined for LPs.

In [ ]:
mip2, _, _, _ = build_model(
    ingredients, meal_ids, B, L, D, p, s, r, a,
    lam=lam, nut_bounds=NUT_BOUNDS_RELAXED
)
mip2.setParam("OutputFlag", 0)
mip2.update()

lp = mip2.relax()
lp.setParam("OutputFlag", 0)
lp.optimize()

lp_obj = lp.ObjVal
print(f"LP objective:  ${lp_obj:.4f}")

## Integrality Gap

In [ ]:
gap = (mip_obj - lp_obj) / abs(lp_obj) * 100
print(f"Integrality gap: {gap:.2f}%")
print(f"  MIP = ${mip_obj:.4f}, LP = ${lp_obj:.4f}, difference = ${mip_obj - lp_obj:.4f}")

gap_df = pd.DataFrame([{
    "mip_objective": mip_obj, "lp_objective": lp_obj,
    "integrality_gap_pct": gap, "gap_dollars": mip_obj - lp_obj,
}])
gap_df.to_csv("results/integrality_gap.csv", index=False)
gap_df

## Shadow Prices — Nutritional Constraints (C8)

For lower-bound constraints: positive shadow price means the LB is binding (nutrient is scarce).  
For upper-bound constraints: negative shadow price means the UB is binding (ceiling hit).

In [ ]:
# Collect shadow prices from all constraints
shadow_data = []
all_shadow_data = []

for constr in lp.getConstrs():
    name = constr.ConstrName
    pi = constr.Pi
    all_shadow_data.append({"constraint": name, "shadow_price": pi})

    if not name.startswith("C8_"):
        continue
    parts = name.split("_")
    direction = parts[1]
    day = int(parts[-1][1:])
    nutrient = "_".join(parts[2:-1])
    shadow_data.append({
        "constraint": name, "direction": direction,
        "nutrient": nutrient, "day": day, "shadow_price": pi,
    })

df_shadow = pd.DataFrame(shadow_data)
df_all = pd.DataFrame(all_shadow_data)

df_shadow.to_csv("results/shadow_prices_by_day.csv", index=False)
df_all.to_csv("results/shadow_prices_all_constraints.csv", index=False)

In [ ]:
# Summarize across days
summary = (
    df_shadow
    .groupby(["nutrient", "direction"])["shadow_price"]
    .agg(["mean", "min", "max", "std"])
    .reset_index()
    .rename(columns={"mean": "avg_shadow", "min": "min_shadow",
                      "max": "max_shadow", "std": "std_shadow"})
)
summary.to_csv("results/shadow_prices_summary.csv", index=False)

# Show binding constraints
binding = summary[summary["avg_shadow"].abs() > 1e-7].copy()
if len(binding) > 0:
    print("Binding nutritional constraints:")
    display(binding)
else:
    print("No nutritional constraints are binding in the LP relaxation.")
    print("All shadow prices on C8 constraints are zero.")

## Shadow Prices — Structural Constraints

These are the meal slot assignments (C1-C3), variety limits (C4-C6), no-consecutive rule (C7), weekly calorie bounds (C9), and waste balance (C11).

In [ ]:
groups = {
    "C1 (breakfast slot)": "C1_",
    "C2 (lunch slot)": "C2_",
    "C3 (dinner slot)": "C3_",
    "C4 (breakfast variety)": "C4_",
    "C5 (lunch variety)": "C5_",
    "C6 (dinner variety)": "C6_",
    "C7 (no-consecutive)": "C7_",
    "C9_lb (weekly cal)": "C9_lb",
    "C9_ub (weekly cal)": "C9_ub",
    "C11 (waste balance)": "C11_",
}

struct_rows = []
for label, prefix in groups.items():
    rows = df_all[df_all["constraint"].str.startswith(prefix)]
    if rows.empty:
        continue
    binding_count = (rows["shadow_price"].abs() > 1e-6).sum()
    struct_rows.append({
        "constraint_group": label,
        "avg_abs_shadow": rows["shadow_price"].abs().mean(),
        "max_abs_shadow": rows["shadow_price"].abs().max(),
        "binding": f"{binding_count}/{len(rows)}",
    })

pd.DataFrame(struct_rows)

## Interpretation

In the LP relaxation, the **structural constraints** (meal slot assignments, variety caps, waste balance) carry the shadow prices, while the **nutritional constraints are all non-binding** (shadow price = 0).

This means the dominant cost driver is the binary meal selection structure — the requirement to pick exactly one complete meal per slot per day — not the nutritional requirements themselves. The LP can trivially satisfy all 14 nutrient bounds by mixing fractional meals optimally.

The large integrality gap confirms this: relaxing integrality dramatically reduces the objective because the LP can blend meals in fractional proportions.